# 🌌 Universal Astrological OKF Engine
This notebook serves as a multi-tenant engine for calculating high-precision astrological charts and converting them into the **Open Knowledge Format (OKF)**—a modular, human-readable system designed for knowledge management tools like Obsidian.

In [ ]:
import sys

# Check for Google Colab environment
is_colab = 'google.colab' in sys.modules

missing_packages = []
for pkg_name, import_name in [
    ("pysweph", "swisseph"),
    ("requests", "requests"),
    ("tzdata", "tzdata"),
    ("jinja2", "jinja2"),
    ("python-dateutil", "dateutil"),
    ("geopy", "geopy")
]:
    try:
        __import__(import_name)
    except ImportError:
        missing_packages.append(pkg_name)

if not missing_packages:
    print("✅ All required packages are installed.")
else:
    print(f"❌ Missing packages: {', '.join(missing_packages)}")
    if is_colab:
        print("Running in Google Colab: Installing missing packages automatically...")
        import subprocess
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
            print("✅ Packages installed successfully. Please restart runtime if needed.")
        except Exception as e:
            print(f"❌ Error installing packages: {e}")
    else:
        print("\nPlease run the following command in your terminal to install them:")
        print(f"    {sys.executable} -m pip install {' '.join(missing_packages)}")
        print("\nAfter installation, please restart the Jupyter kernel in VS Code.")

### 📦 Libraries and Dependencies
This block imports the necessary Python standard libraries and external packages.
- `swisseph` (as `swe`) is the core engine for astronomical calculations.
- `yaml`, `json`, and `hashlib` are used for data formatting and file namespacing.
- `zoneinfo` and `datetime` handle high-precision time conversions.

In [ ]:
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# --- STANDARD LIBRARY IMPORTS ---
import json
import hashlib
import shutil
from datetime import datetime, timezone, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

# --- THIRD-PARTY IMPORTS ---
import requests
import swisseph as swe


## 🛰️ Ephemeris Data Acquisition
To ensure scientific-grade precision, we utilize the **Swiss Ephemeris**. This section downloads the necessary planetary data files (`.se1`) from the official repository. These files are required for calculating 'True' planetary positions and node calculations.

In [ ]:
# 1. Create ephemeris directory
ephe_path = Path('ephe').resolve()
ephe_path.mkdir(parents=True, exist_ok=True)

# 2. Correct raw URLs
base_url = "https://github.com/aloistr/swisseph/raw/master/ephe/"
urls = [
    base_url + "seas_18.se1",
    base_url + "sepl_18.se1",
    base_url + "semo_18.se1"
]

for url in urls:
    file_name = url.split('/')[-1]
    target = ephe_path / file_name

    # Cleanup existing corrupt/small files
    if target.exists() and target.stat().st_size < 5000:
        target.unlink()

    if not target.exists():
        print(f"Downloading {file_name}...")
        try:
            r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            if r.status_code == 200:
                target.write_bytes(r.content)
                print(f"Successfully downloaded {file_name}")
            else:
                print(f"Failed to download {file_name}: Status {r.status_code}")
        except Exception as e:
            print(f"Error downloading {file_name}: {e}")
    else:
        print(f"✅ {file_name} already exists. Skipping download.")

# 3. Configure and Verify
swe.set_ephe_path(str(ephe_path))
files_info = {p.name: f"{p.stat().st_size / 1024:.2f} KB" for p in ephe_path.glob('*.se1')}
print(f"Ephemeris path set to: {ephe_path}")
print(f"Verified Files: {files_info}")

### 🛠️ Configuration and User Inputs
The form below allows you to define the subject's birth data (Date, Time, Location) and specific astrological settings like the **Ayanamsa** (sidereal offset) and **House System**. These variables drive the entire calculation engine.

## ⚙️ Profile Configuration
Use the form below to set the subject's birth data.
- **Subject Name**: Used for namespacing directories and files.
- **Coordinates**: High-precision latitude/longitude is required for accurate calculations.
  * *Automatic Geocoding*: You can set `AUTO_GEOLOCATE = True` and enter the `BIRTH_PLACE` (e.g. `"San Francisco, CA"` or `"Paris, France"`) to automatically retrieve coordinates using `geopy` / OpenStreetMap.
  * *Manual Lookup*: If offline or disabling auto-geolocating, search on [LatLong.net](https://www.latlong.net) or [Google Maps](https://www.google.com/maps) (right-click location to copy coordinates) and input them manually.
- **Settings**: Choose your preferred Ayanamsa (Sidereal Offset) and House System (e.g., Whole Sign or Placidus).

### 🧠 Core Astrological Engine
This section defines the mathematical and logical framework for the chart:
- `AstroTime`: Converts local time to Julian Day (JD) and handles UTC offsets.
- `AstroMath`: Provides utility functions for mapping longitudes to signs and nakshatras.
- `VedicAspectEngine`: Logic for calculating planetary sight (Graha Drishti).
- `FriendshipEngine`: Calculates natural and temporary relationships between planets (Panchadha Maitri).
- `DignityEngine`: Determines planetary strength (Exaltation, Moolatrikona, etc.).
- `VimshottariDashaEngine`: Projects time-based planetary cycles based on the Moon's position.

In [ ]:
# @title ⚙️ Birth Data Configuration

SUBJECT_NAME = "Steve Jobs" # @param {type:"string"}
B_YEAR = 1955 # @param {type:"integer"}
B_MONTH = 2 # @param {type:"integer"}
B_DAY = 24 # @param {type:"integer"}
B_TIME = "19:15" # @param {type:"string"}
B_TZ = "America/Los_Angeles" # @param {type:"string"}
GENDER = "Male" # @param ["Male", "Female"]

# @markdown **Geolocation Settings:**
BIRTH_PLACE = "San Francisco, CA" # @param {type:"string"}
AUTO_GEOLOCATE = True # @param {type:"boolean"}

# @markdown Manual coordinates (used if AUTO_GEOLOCATE is False or lookup fails):
LATITUDE = 37.7749 # @param {type:"number"}
LONGITUDE = -122.4194 # @param {type:"number"}

# @markdown ---
# @markdown ### Astrological Settings
# @markdown **Report Mode:** Controls what data is included in the OKF output.
# @markdown - `Tropical-Vedic`: Tropical Signs/Houses + Vedic Nakshatras/Dashas/Aspects.
REPORT_MODE = "Tropical-Vedic" # @param ["Tropical Focus", "Vedic Focus", "Everything", "Tropical-Vedic"]

HOUSE_SYSTEM = "W" # @param ["W", "P", "E"]
AYANAMSA = 36 # @param [36, 1, 3, 5]

# @markdown **Export Settings:**
# @markdown - If False, all files stay organized in  without extra zip files.
CREATE_ZIP_ARCHIVE = False # @param {type:"boolean"}

# --- AUTO-COMPUTED EXPORT PATHS & METADATA ---
import os
import hashlib

# Perform automatic geolocation if requested
if AUTO_GEOLOCATE and BIRTH_PLACE:
    try:
        from geopy.geocoders import Nominatim
        geolocator = Nominatim(user_agent="astrology_engine")
        location = geolocator.geocode(BIRTH_PLACE)
        if location:
            LATITUDE = round(location.latitude, 4)
            LONGITUDE = round(location.longitude, 4)
            print(f"✅ Successfully geocoded '{BIRTH_PLACE}' to Lat: {LATITUDE}, Lon: {LONGITUDE}")
        else:
            print(f"⚠️ Could not geolocate '{BIRTH_PLACE}'. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")
    except Exception as e:
        print(f"⚠️ Geolocation query failed: {e}. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")
else:
    print(f"ℹ️ Auto-geolocation skipped. Using manual coordinates (Lat: {LATITUDE}, Lon: {LONGITUDE}).")

safe_subject = SUBJECT_NAME.lower().replace(" ", "_")
target_json_path = str(Path("profiles") / safe_subject / f"{safe_subject}_astrology_profile.json")

# Generate unique fingerprint based on input parameters
fingerprint_input = f"{SUBJECT_NAME}_{B_YEAR}_{B_MONTH}_{B_DAY}_{B_TIME}_{LATITUDE}_{LONGITUDE}_{HOUSE_SYSTEM}_{AYANAMSA}"
fingerprint = hashlib.md5(fingerprint_input.encode('utf-8')).hexdigest()[:8]

subject_profile = {
    "name": SUBJECT_NAME,
    "gender": GENDER,
    "birth_data": {
        "date": f"{B_YEAR}-{B_MONTH:02d}-{B_DAY:02d}",
        "time": B_TIME,
        "timezone": B_TZ,
        "latitude": LATITUDE,
        "longitude": LONGITUDE
    }
}

HOUSE_SYSTEM_NAMES = {"P": "Placidus", "E": "Equal", "W": "Whole Sign"}
AYANAMSA_NAMES = {
    36: "Galactic Center at 0 Mula (Wilhelm)",
    1: "Lahiri",
    3: "Raman",
    0: "Fagan-Bradley",
    5: "KP"
}

metadata = {
    "engine": "Universal Astrological OKF Engine",
    "version": "2.0",
    "settings": {
        "report_mode": REPORT_MODE,
        "house_system": HOUSE_SYSTEM,
        "house_system_name": HOUSE_SYSTEM_NAMES.get(HOUSE_SYSTEM, "Whole Sign"),
        "ayanamsa": AYANAMSA,
        "ayanamsa_name": AYANAMSA_NAMES.get(AYANAMSA, f"Custom ({AYANAMSA})")
    }
}

# 🔱 Astrological Calculation Engines & Exporter

This section defines the unified calculation engines for computing divisional harmonics, planetary strength arrays, auspicious point distributions, and planetary yoga configurations. All engines dynamically adapt to the active `REPORT_MODE` (Tropical vs. Vedic coordinate frames).

### ⚙️ Engine Summary
1. **`AstroMath`**: Core utilities for sign, nakshatra, and house placements.
2. **`VargaEngine`**: Computes harmonic divisions (D1 through D60) based on active longitudes.
3. **`ShadbalaEngine`**: Calculates planetary strength across 6 categories (Sthana, Dig, Kala, Cheshta, Naisargika, Drik).
4. **`AshtakavargaEngine`**: Computes Bhinna (BAV) and Sarvashtakavarga (SAV) point matrices.
5. **`YogaEngine`**: Scans the birth chart for classic planetary yogas.
6. **`GocharaEngine`**: Evaluates transit alignments relative to the natal Moon position.
7. **`OKFConverter`**: Outputs structured, Obsidian-linked Markdown reports.

In [ ]:
# @title 🔱 Astrological Calculation Engines & Exporter
%load_ext autoreload
%autoreload 2

from astrology_engine import *


# 🚀 Calculation & Exporter Execution Pipeline

This cell executes the calculations sequentially:
1. Computes natal houses and instantiates planetary coordinates.
2. Caches house numbers on the planet objects directly to optimize friendship and divisional friendship loops.
3. Runs the complete Shadbala strength calculations, Ashtakavarga point grids, Yoga scanner, and Gochara transit checks.
4. Exports the structured results to JSON and runs the Obsidian converter.

> [!TIP]
> **Google Colab restoration**:
> To run this notebook in Google Colab, restore the Swiss Ephemeris path setting:
> `ephe_path = '/content/ephe'`

In [ ]:
try:
    # 1. Execute Unified Calculation Pipeline via AstrologyPipeline
    yogas_rules_path = Path("yogas").resolve()
    chart_result = AstrologyPipeline.calculate(
        year=B_YEAR, month=B_MONTH, day=B_DAY, hour=B_TIME, tz=B_TZ,
        lat=LATITUDE, lon=LONGITUDE, mode=REPORT_MODE,
        house_system=HOUSE_SYSTEM, ayanamsa=AYANAMSA,
        yogas_rules_path=yogas_rules_path,
        subject=subject_profile,
        compute_transits=True
    )

    # 2. Export All Artifacts (Profile JSON, SLM Markdown, Lean JSON, HTML Report)
    target_json = Path(target_json_path)
    output_dir = target_json.parent
    template_path = Path("report_template.html").resolve()

    artifacts = chart_result.export_all(
        output_dir=output_dir,
        filename_prefix=safe_subject,
        template_path=template_path,
        metadata=metadata,
        fingerprint=fingerprint
    )

    target_json = artifacts["json"]
    target_md_path = artifacts["markdown"]
    target_llm_json_path = artifacts["lean_json"]
    report_output_path = artifacts["html"]
    planets_data = chart_result.planets_data

    print(f"✅ Calculation Pipeline Success. Data saved to: {target_json}")
    print(f"✅ SLM-optimized Markdown context created at: {target_md_path}")
    print(f"✅ Lean JSON payload saved to: {target_llm_json_path}")
    print(f"✅ HTML Report successfully generated at: {report_output_path}")
    print(f"Mode: {REPORT_MODE} | Divisional Data: {list(chart_result.planets_data[0].get('vargas', {}).keys())}")

except Exception as e:
    import traceback
    print(f"❌ Execution Error: {e}")
    traceback.print_exc()

In [ ]:
import json
import shutil
from pathlib import Path

# 1. Verify Varga Content Integrity (D9, D10, D12, D60) from JSON
subject_dir = Path('profiles') / safe_subject
target_path = Path(target_json_path)

with target_path.open('r', encoding='utf-8') as f:
    json_data = json.load(f)

first_planet_vargas = json_data['astrological_chart']['planets'][0].get('vargas', {})
vargas_checks = {v: (v in first_planet_vargas) for v in ['D9', 'D10', 'D12', 'D60']}

# Clean up any legacy 'charts' directory in output
charts_dir = subject_dir / 'charts'
if charts_dir.exists():
    shutil.rmtree(charts_dir)

# 2. Verify Output Files (Markdown Context, Lean JSON & HTML Report)
md_file = subject_dir / f"{safe_subject}_llm_context.md"
md_exists = md_file.exists() and md_file.stat().st_size > 0

json_file = subject_dir / f"{safe_subject}_llm_payload.json"
json_exists = json_file.exists() and json_file.stat().st_size > 0

report_file = subject_dir / f"{safe_subject}_report.html"
report_exists = report_file.exists() and report_file.stat().st_size > 0

# 3. Optional Zip Archive (only if requested or running in Google Colab)
should_zip = globals().get('CREATE_ZIP_ARCHIVE', False) or globals().get('is_colab', False)
archive_msg = f"Skipped (all outputs available in {subject_dir})"

if should_zip:
    final_zip_name = f"{safe_subject}_astrology_profile_bundle"
    archive_path = shutil.make_archive(str(subject_dir / final_zip_name), 'zip', str(subject_dir))
    archive_msg = f"Created at {archive_path}"

print("--- Final Production Verification ---")
print(f"✅ Calculation Integrity Verified from JSON for: {SUBJECT_NAME}")
print(f"✅ Varga Integrity Checked: {vargas_checks}")
print(f"✅ SLM Markdown Context Verified: {md_exists}")
print(f"✅ Lean JSON Payload Verified: {json_exists}")
print(f"✅ HTML Report Generated & Verified: {report_exists}")
print(f"📦 Archive Status: {archive_msg}")
print(f"📁 Output Directory: {subject_dir.resolve()}")
print("\nThe engine is now production-ready with HTML dashboard and SLM Markdown context output.")